### **Gaussian Splatting for Hayden Library Lantern**

In [ ]:
!git clone https://github.com/ekagra1602/gaussian-splatting-spectral
%cd gaussian-splatting-spectral

Cloning into 'gaussian-splatting-spectral'...
remote: Enumerating objects: 228, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 228 (delta 1), reused 11 (delta 1), pack-reused 214 (from 1)
Receiving objects: 100% (228/228), 12.87 MiB | 38.21 MiB/s, done.
Resolving deltas: 100% (104/104), done.
/content/gaussian-splatting-spectral


In [ ]:
!pip install -r requirements.txt
!apt-get -qq update && apt-get -qq install -y ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 19.0 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


**Prep Dataset**

In [ ]:
#Renew Dataset
!rm -rf {/content/gaussian-splatting-spectral/lantern_ds_full}

In [ ]:
%cd /content/gaussian-splatting-spectral
!python src/dataset_prep.py \
  --video /content/knowledge_lantern.mp4 \
  --out lantern_ds_full \
  --target_frames 120 \
  --min_sharpness 70 \
  --width 1600

/content/gaussian-splatting-spectral
>> ffmpeg -y -i /content/knowledge_lantern.mp4 -vf fps=15 -q:v 1 lantern_ds_full/frames_raw/frame_%05d.jpg
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enab

## **Run VGGT to predict camera poses, depth**

In [ ]:
!git clone https://github.com/facebookresearch/vggt.git /content/vggt
%cd /content/vggt
!python -c "import vggt; print('VGGT installed!')"

Cloning into '/content/vggt'...
remote: Enumerating objects: 1265, done.
remote: Total 1265 (delta 0), reused 0 (delta 0), pack-reused 1265 (from 1)
Receiving objects: 100% (1265/1265), 64.94 MiB | 39.10 MiB/s, done.
Resolving deltas: 100% (579/579), done.
/content/vggt
VGGT installed!


In [ ]:
!pip install -r requirements_demo.txt

  Cloning https://github.com/jytime/LightGlue.git to /tmp/pip-install-l_8_bbvn/lightglue_c7c6ded5b3b64d3c83ced507ce5920bf
  Running command git clone --filter=blob:none --quiet https://github.com/jytime/LightGlue.git /tmp/pip-install-l_8_bbvn/lightglue_c7c6ded5b3b64d3c83ced507ce5920bf
  Resolved https://github.com/jytime/LightGlue.git to commit 2f23ca2ea9638cecad7f7220795210fc6b8353c3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.9/28.9 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.7/431.7 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 24.8 MB/s eta 0:00:00
  

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
!python /content/vggt/demo_colmap.py \
  --scene_dir=/content/gaussian-splatting-spectral/lantern_ds_full \
  --use_ba \
  --query_frame_num=8 \
  --max_query_pts=3072 \
  --fine_tracking \
  --vis_thresh=0.2

/usr/local/lib/python3.12/dist-packages/lightglue/lightglue.py:24: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
Arguments: {'scene_dir': '/content/gaussian-splatting-spectral/lantern_ds_full', 'seed': 42, 'use_ba': True, 'max_reproj_error': 8.0, 'shared_camera': False, 'camera_type': 'SIMPLE_PINHOLE', 'vis_thresh': 0.2, 'query_frame_num': 8, 'max_query_pts': 3072, 'fine_tracking': True, 'conf_thres_value': 5.0}
Setting seed as: 42
Using device: cuda
Using dtype: torch.bfloat16
Downloading: "https://huggingface.co/facebook/VGGT-1B/resolve/main/model.pt" to /root/.cache/torch/hub/checkpoints/model.pt
100% 4.68G/4.68G [00:08<00:00, 606MB/s]
Model loaded
Loaded 120 images from /content/gaussian-splatting-spectral/lantern_ds_full/images
/content/vggt/demo_colmap.py:75: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use

## **Restore Session**

In [ ]:
# restore_session.py - Start of every new session
import os
from google.colab import drive

print("🔄 Restoring session...")

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Smart repo handling
REPO_URL = "https://github.com/ekagra1602/gaussian-splatting-spectral"
REPO_DIR = "/content/gaussian-splatting-spectral"

if not os.path.exists(REPO_DIR):
    print("📥 Cloning repository (first time)...")
    !git clone {REPO_URL}
else:
    print("✅ Repository exists, pulling latest changes...")
    %cd {REPO_DIR}
    !git pull origin main || echo "⚠️  Pull failed (maybe no internet or no changes)"

%cd {REPO_DIR}

# 3. Restore dataset from Google Drive backup
print("📦 Restoring lantern_ds dataset...")
if os.path.exists('/content/drive/MyDrive/GaussianSplatting_Backup/lantern_ds'):
    !cp -r /content/drive/MyDrive/GaussianSplatting_Backup/lantern_ds {REPO_DIR}/
    print("✅ Dataset restored from Drive")
else:
    print("❌ Backup not found in Drive! Check your backup location.")

# 4. Reinstall dependencies
print("📚 Installing dependencies...")
!pip install -q hydra-core omegaconf pycolmap==3.10.0

# 5. Comprehensive verification
print("\n🔍 Verification:")
checks = {
    "Repository": os.path.exists(REPO_DIR),
    "lantern_ds": os.path.exists(f"{REPO_DIR}/lantern_ds"),
    "images": os.path.exists(f"{REPO_DIR}/lantern_ds/images"),
    "sparse": os.path.exists(f"{REPO_DIR}/lantern_ds/sparse"),
    "COLMAP files": os.path.exists(f"{REPO_DIR}/lantern_ds/sparse/cameras.bin")
}

for item, exists in checks.items():
    status = "✅" if exists else "❌"
    print(f"  {status} {item}")

if checks["sparse"]:
    sparse_files = os.listdir(f"{REPO_DIR}/lantern_ds/sparse/")
    print(f"\n📁 Sparse folder contains: {sparse_files}")

print(f"\n📂 Working directory: {os.getcwd()}")
print("\n🚀 Session fully restored! Ready to continue.")

### Backup Session

In [ ]:
# 💾 BACKUP CELL
from google.colab import drive
import os

# Mount Drive (safe to run multiple times)
drive.mount('/content/drive')

WORK_DIR = "/content/GaussianSplatting/lantern_ds"
BACKUP_DIR = "/content/drive/MyDrive/gaussian-splatting-spectral_backup"

print("💾 Starting backup...")

# Create backup directory
!mkdir -p {BACKUP_DIR}

# Sync with progress
!rsync -av --progress {WORK_DIR}/ {BACKUP_DIR}/lantern_ds/

# Show what was backed up
print("\n📊 Backup Summary:")
!du -sh {BACKUP_DIR}/lantern_ds/
print(f"📁 Backed up to: {BACKUP_DIR}/lantern_ds/")

# List major folders
print("\n📂 Backed up folders:")
for folder in ['images', 'sparse', 'outputs', 'checkpoints']:
    folder_path = f"{BACKUP_DIR}/lantern_ds/{folder}"
    if os.path.exists(folder_path):
        size = !du -sh {folder_path} | cut -f1
        print(f"  ✅ {folder}: {size[0]}")
    else:
        print(f"  ⚪ {folder}: (not yet created)")

print("\n✅ Backup complete! Safe to disconnect.")

# **Run Gaussian Splatting on output from VGGT**

In [ ]:
# Install gsplat (recommended version from VGGT docs)
!pip install gsplat

# Install additional dependencies
!pip install \
  opencv-python \
  pillow \
  tqdm \
  lpips \
  trimesh \
  plyfile

print("✅ gsplat installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.3 MB/s eta 0:00:00
✅ gsplat installed!


In [ ]:
# Clone gsplat repo to get example training scripts
%cd /content
!git clone https://github.com/nerfstudio-project/gsplat.git
%cd gsplat

# Install example requirements
!pip install -r examples/requirements.txt

print("✅ gsplat repo cloned!")

/content
Cloning into 'gsplat'...
remote: Enumerating objects: 10424, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 10424 (delta 25), reused 13 (delta 11), pack-reused 10381 (from 4)
Receiving objects: 100% (10424/10424), 133.90 MiB | 39.81 MiB/s, done.
Resolving deltas: 100% (7117/7117), done.
/content/gsplat
  Cloning https://github.com/rmbrualla/pycolmap (to revision cc7ea4b7301720ac29287dbe450952511b32125e) to /tmp/pip-req-build-7fmqw9qm
  Running command git clone --filter=blob:none --quiet https://github.com/rmbrualla/pycolmap /tmp/pip-req-build-7fmqw9qm
  Running command git rev-parse -q --verify 'sha^cc7ea4b7301720ac29287dbe450952511b32125e'
  Running command git fetch -q https://github.com/rmbrualla/pycolmap cc7ea4b7301720ac29287dbe450952511b32125e
  Resolved https://github.com/rmbrualla/pycolmap to commit cc7ea4b7301720ac29287dbe450952511b32125e
  Installing build dependencies ... done
  Getting requirements

✅ gsplat repo cloned!


In [ ]:
#Create 0 directory
import os
import shutil

# Create the 0 subdirectory
sparse_dir = "/content/gaussian-splatting-spectral/lantern_ds_full/sparse"
sparse_0_dir = f"{sparse_dir}/0"

os.makedirs(sparse_0_dir, exist_ok=True)

# Move all .bin and .ply files into 0/
for file in os.listdir(sparse_dir):
    if file.endswith(('.bin', '.ply')):
        src = f"{sparse_dir}/{file}"
        dst = f"{sparse_0_dir}/{file}"
        shutil.move(src, dst)
        print(f"Moved: {file} → sparse/0/{file}")

print("\n✅ Dataset structure fixed!")

Moved: images.bin → sparse/0/images.bin
Moved: cameras.bin → sparse/0/cameras.bin
Moved: points.ply → sparse/0/points.ply
Moved: points3D.bin → sparse/0/points3D.bin

✅ Dataset structure fixed!


In [ ]:
!echo "=== Checking dataset structure ==="
!ls -lh /content/gaussian-splatting-spectral/lantern_ds/
!echo ""
!ls -lh /content/gaussian-splatting-spectral/lantern_ds/sparse/0/
!echo ""
!ls /content/gaussian-splatting-spectral/lantern_ds/images/ | head -5

=== Checking dataset structure ===
total 4.0K
drwxr-xr-x 3 root root 4.0K Nov 21 04:02 sparse

total 0

ls: cannot access '/content/gaussian-splatting-spectral/lantern_ds/images/': No such file or directory


### ***Run Custom Script***

In [ ]:
# Uninstall current pycolmap
!pip uninstall pycolmap -y

# Install the EXACT version gsplat needs (from their requirements.txt)
!pip install git+https://github.com/rmbrualla/pycolmap@cc7ea4b7301720ac29287dbe450952511b32125e

# Verify it worked
!python -c "from pycolmap import SceneManager; print('✅ SceneManager imported successfully')"

  Cloning https://github.com/rmbrualla/pycolmap (to revision cc7ea4b7301720ac29287dbe450952511b32125e) to /tmp/pip-req-build-q2c6mhy7
  Running command git clone --filter=blob:none --quiet https://github.com/rmbrualla/pycolmap /tmp/pip-req-build-q2c6mhy7
  Running command git rev-parse -q --verify 'sha^cc7ea4b7301720ac29287dbe450952511b32125e'
  Running command git fetch -q https://github.com/rmbrualla/pycolmap cc7ea4b7301720ac29287dbe450952511b32125e
  Resolved https://github.com/rmbrualla/pycolmap to commit cc7ea4b7301720ac29287dbe450952511b32125e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pycolmap: filename=pycolmap-0.0.1-py3-none-any.whl size=16590 sha256=8a703632c858b176ba17a57a147e72828b2125b9290e2805dfd395507d45a6be
  Stored in directory: /root/.cache/pip/wheels/a0/41/d4/6cf63b78fbaab3a441b7d6032c076261bc9c16d2f50172f1a3
Successfully built pycolmap
✅ SceneManager impor

In [17]:
!pip uninstall -y datasets

Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0


In [18]:
import os, shutil

SRC = "/content/gaussian-splatting-spectral/src"
EXP = "/content/gsplat/examples"

# Copy custom training script
shutil.copy(
    os.path.join(SRC, "gaussian_spectral_training.py"),
    os.path.join(EXP, "gaussian_spectral_training.py"),
)

# Copy spectral loss helper
shutil.copy(
    os.path.join(SRC, "spectral_loss.py"),
    os.path.join(EXP, "spectral_loss.py"),
)

print("✅ Copied gaussian_spectral_training.py and spectral_loss.py into gsplat/examples")

✅ Copied gaussian_spectral_training.py and spectral_loss.py into gsplat/examples


In [25]:
import os
import shutil
import urllib.request
import zipfile

# Define the target directory for GLM
glm_dir = "/content/gsplat/gsplat/cuda/csrc/third_party/glm"
check_file = os.path.join(glm_dir, "glm", "gtc", "type_ptr.hpp")

# Check if GLM is already installed
if not os.path.exists(check_file):
    print("⚠️ GLM headers missing. Downloading...")

    # Clean up existing empty directory if needed
    if os.path.exists(glm_dir):
        shutil.rmtree(glm_dir)

    # Download GLM
    url = "https://github.com/g-truc/glm/archive/refs/tags/0.9.9.8.zip"
    zip_path = "/content/glm.zip"
    urllib.request.urlretrieve(url, zip_path)

    # Extract
    print("Extracting...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("/content")

    # Move to the correct location
    # The zip extracts to "glm-0.9.9.8", we rename it to "glm" and move it to the third_party dir
    source_path = "/content/glm-0.9.9.8"
    os.makedirs(os.path.dirname(glm_dir), exist_ok=True)
    shutil.move(source_path, glm_dir)

    # Cleanup
    os.remove(zip_path)
    print(f"✅ GLM installed to {glm_dir}")
else:
    print("✅ GLM headers already present.")

⚠️ GLM headers missing. Downloading...
Extracting...
✅ GLM installed to /content/gsplat/gsplat/cuda/csrc/third_party/glm


In [40]:
# ================================================
# CELL 3: Train with gaussian_spectral_training.py
# ================================================
%cd /content/gsplat/examples

import torch, gc
torch.cuda.empty_cache()
gc.collect()

!python gaussian_spectral_training.py default \
    --data_dir /content/gaussian-splatting-spectral/lantern_ds_full \
    --result_dir /content/results/lantern_spectral_lambda \
    --max_steps 10000 \
    --spectral_lambda 0.05


/content/gsplat/examples
[Parser] 120 images, taken by 120 cameras.
Scene scale: 1.547637936955762
Model initialized. Number of GS: 25200
  0% 0/10000 [00:00<?, ?it/s]W1121 06:21:15.280000 43021 torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W1121 06:21:15.280000 43021 torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.
step=599 loss=0.0603 photo=0.0596spec=0.0154 H=0.985 sh=0:   6% 592/10000 [00:08<01:52, 83.37it/s]Step 600: 795 GSs duplicated, 367 GSs split. Now having 26362 GSs.
Step 600: 102 GSs pruned. Now having 26260 GSs.
step=699 loss=0.0623 photo=0.0612spec=0.0222 H=0.978 sh=0:   7% 699/10000 [00:09<01:45, 88.02it/s]Step 700: 2159 GSs duplicated, 418 GSs split. Now having 28837 GSs.
Step 700: 70 GSs pruned. Now having 28767 GSs.
step=799 loss=0.0538 photo=0.0524spec=0.0288 H=0.971 sh=0:   8% 799/10000 [00:10<01:49, 84.3

In [44]:
# 1. Run Eval on Baseline
!python /content/gsplat/examples/simple_trainer.py default \
    --data_dir /content/gaussian-splatting-spectral/lantern_ds_full \
    --result_dir /content/results/lantern_spectral \
    --ckpt /content/results/lantern_baseline/ckpts/train_step9999.pt \
    --disable_viewer \
    --data_factor 1

2025-11-21 06:51:58.373938: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-21 06:51:58.392289: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763707918.413919   50939 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763707918.420418   50939 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1763707918.436991   50939 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [45]:
# 2. Run Eval on Spectral Model
!python /content/gsplat/examples/simple_trainer.py default \
    --data_dir /content/gaussian-splatting-spectral/lantern_ds_full \
    --result_dir /content/results/lantern_spectral_lambda \
    --ckpt /content/results/lantern_spectral_lambda/ckpts/train_step9999.pt \
    --disable_viewer \
    --data_factor 1

2025-11-21 06:59:00.649773: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-21 06:59:00.667783: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763708340.689136   52872 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763708340.695632   52872 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1763708340.712228   52872 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [41]:
#Download PLY File
from google.colab import files
files.download("/content/results/lantern_spectral_lambda/ply/train_step9999.ply")
files.download("/content/results/lantern_spectral_lambda/ckpts/train_step9999.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
!pip install pyngrok
from pyngrok import ngrok
ngrok.set_auth_token("35m6AuJLaeavG43ZdnzxIry5XZc_igNGYXdGZujTXgb9TPPt")
# Open a tunnel to port 8081 (Viser's default is often 8080, but your log says 8081)
public_url = ngrok.connect(8081).public_url
print(f"🚀 CLICK THIS LINK TO VIEW: {public_url}")

🚀 CLICK THIS LINK TO VIEW: https://aerodynamically-pulsatile-janiya.ngrok-free.dev


In [33]:
!python /content/gsplat/examples/simple_viewer.py --ckpt /content/results/lantern_spectral/ckpts/train_step9999.pt --port 8080

Number of Gaussians: 210737
╭──────────────── viser ────────────────╮
│             ╷                         │
│   HTTP      │ http://localhost:8081   │
│   Websocket │ ws://localhost:8081     │
│             ╵                         │
╰───────────────────────────────────────╯
Viewer running... Ctrl+C to exit.
W1121 05:27:56.366000 29538 torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W1121 05:27:56.366000 29538 torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.
object address  : 0x7e08e8bb57e0
object refcount : 3
object type     : 0xa2a4e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
connection handler failed
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/viser/infra/_infra.py", line 368, in ws_handler
    await asyncio.gather(
  File "/usr/local

In [ ]:

# ============================================================
# CELL 4: Check Results
# ============================================================
import os

result_dir = "/content/GaussianSplatting/results/lantern_spectral_30k"

print("📊 Training Results\n")
print("=" * 60)

if os.path.exists(result_dir):
    # List all files
    files = os.listdir(result_dir)

    # Count checkpoints
    checkpoints = [f for f in files if f.endswith('.pt')]
    print(f"✅ Checkpoints saved: {len(checkpoints)}")

    # Check for final model
    if os.path.exists(f"{result_dir}/final.pt"):
        size = os.path.getsize(f"{result_dir}/final.pt") / (1024*1024)
        print(f"✅ Final model: {size:.1f} MB")

    if os.path.exists(f"{result_dir}/final.ply"):
        size = os.path.getsize(f"{result_dir}/final.ply") / (1024*1024)
        print(f"✅ Final PLY: {size:.1f} MB")

    print(f"\n📁 Full results at: {result_dir}")
else:
    print("❌ Results directory not found!")

print("=" * 60)

In [ ]:
# ============================================================
# CELL 5: Download Results
# ============================================================
from google.colab import files

# Download final PLY file for viewing
ply_path = "/content/GaussianSplatting/results/lantern_spectral_30k/final.ply"

if os.path.exists(ply_path):
    print(f"📥 Downloading: {ply_path}")
    files.download(ply_path)
    print("✅ Download complete!")
else:
    print(f"❌ PLY file not found at: {ply_path}")
    print("\nAvailable files:")
    !ls -lh /content/GaussianSplatting/results/lantern_spectral_30k/

In [ ]:
# ============================================================
# CELL 6: Backup to Google Drive (RECOMMENDED)
# ============================================================
from google.colab import drive

# Mount Drive if not already mounted
drive.mount('/content/drive')

BACKUP_DIR = "/content/drive/MyDrive/GaussianSplatting_Backup"
SOURCE_DIR = "/content/GaussianSplatting/results"

print("💾 Backing up results to Google Drive...")
!mkdir -p {BACKUP_DIR}
!rsync -av --progress {SOURCE_DIR}/ {BACKUP_DIR}/results/

print("\n✅ Backup complete!")
print(f"📁 Saved to: {BACKUP_DIR}/results/")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/gsplat

!python examples/simple_trainer.py default \
  --data_dir /content/gaussian-splatting-spectral/lantern_ds_full \
  --result_dir /content/results/lantern_baseline \
  --data-factor 1 \
  --max-steps 30000 \
  --save-ply \
  --disable-viewer \
  --sh-degree 3 \
  --sh-degree-interval 1000 \
  --init-opa 0.1 \
  --init-scale 1.0 \
  --ssim-lambda 0.2 \
  --random-bkgd \
  --strategy.refine-start-iter 500 \
  --strategy.refine-stop-iter 15000 \
  --strategy.reset-every 3000 \
  --strategy.refine-every 100 \
  --strategy.prune-opa 0.005 \
  --strategy.grow-grad2d 0.0002 \
  --strategy.grow-scale3d 0.01 \
  --strategy.prune-scale3d 0.1 \
  --strategy.prune-scale2d 0.15

In [ ]:
%cd /content/gsplat

# See parameters for the 'default' config
!python examples/simple_trainer.py default --help

### **Ignore**

In [ ]:
# Create the missing __init__.py file
!touch /content/gsplat/examples/datasets/__init__.py

print("Created __init__.py in datasets/")

In [ ]:
# Uninstall old version
!pip uninstall -y gsplat

# Install latest version
!pip install gsplat

print("✅ gsplat upgraded to latest version")

In [ ]:
%cd /content/gsplat/examples

!python simple_trainer.py default \
  --data_dir /content/GaussianSplatting/lantern_ds \
  --result_dir /content/GaussianSplatting/results/lantern_baseline \
  --data_factor 1 \
  --max_steps 10000 \
  --save_ply

In [ ]:
!echo "=== PLY Files ==="
!ls -lh /content/GaussianSplatting/results/lantern_baseline/ply/

!echo ""
!echo "=== Rendered Videos ==="
!ls -lh /content/GaussianSplatting/results/lantern_baseline/videos/

In [ ]:
#Download PLY File
from google.colab import files
files.download("/content/GaussianSplatting/results/lantern_spectral_30k/final.pt")